In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt

In [ ]:
results = pd.read_csv('data_storage_8590.csv')
display(results)

In [ ]:
def clipper(val):
    eps = 10**(-5)
    if val > 1:
        return 1-eps
    elif val < 0:
        return 0+eps
    else:
        return val
    
results['clip_pred']=results.apply(lambda row: clipper(row.pred),axis=1)
results.head()

In [ ]:
sns.boxplot(data=results,x='horizon',y='AIC',hue='model_name')

In [ ]:
# results_list=["3nieval/8590.csv","3nieval/8084.csv","3nieval/7079.csv","3nieval/6069.csv","2nieval/6090.csv","4nieval/6090.csv","2ieval/6090.csv","3ieval/6090.csv",
#               "4ieval/6090.csv"]
results_list = ["model_eval/2ni_pred_covar_no.csv","model_eval/1ni_pred_covar_no.csv","model_eval/1ni_pred_covar_no_lowvar.csv","model_eval/2ni_pred_covar_no_lowvar.csv",
                "model_eval/2ni_pred_covar_no_lowvar_05.csv","model_eval/2ni_pred_covar_no_lowvar_fit.csv"]
imports = []
for file in results_list:
    loaded_csv = pd.read_csv(file)
    imports.append(loaded_csv)
results = pd.concat(imports,ignore_index=True)
#results = pd.read_csv('3nieval/8590.csv')
display(results)

In [ ]:
def clipper(val):
    eps = 10**(-7)
    if val > 1-eps:
        return 1-eps
    elif val < eps:
        return 0+eps
    else:
        return val
    
results['clip_pred']=results.apply(lambda row: clipper(row.pred),axis=1)
results.head()

In [ ]:
results.query("model_name=='2ni_lowvar_05'").head()

In [ ]:
horizons = results.train_horizon.unique()
print(horizons)
models = results.model_name.unique()
print(models)
subjects = results.id.unique()

proc_list = []
for model in models:
    for horizon in horizons:
        filt_df = results.query("train_horizon==@horizon & model_name == @model")
        count = filt_df.shape[0]
        true_val = filt_df.actual.to_numpy()
        count_lapse = sum(true_val)
        pred_val = filt_df.clip_pred.to_numpy()
        auc_score = metrics.roc_auc_score(true_val,pred_val)
        aucpr_score = metrics.average_precision_score(true_val,pred_val)
        iter_df = pd.DataFrame({'horizon':horizon,'model':model,'auc':auc_score,'aucpr':aucpr_score},index=[0])
        proc_list.append(iter_df)
        fig,ax = plt.subplots(1,2,figsize=(12,5))
        fig.suptitle('Training horizon: {}, Model: {}, Test count: {}, Lapse fraction: {:.2f}'.format(horizon,model,count,count_lapse/count))
        metrics.RocCurveDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax[0])
        metrics.PrecisionRecallDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax[1])

proc_results=pd.concat(proc_list,ignore_index=True)

In [ ]:
#horizons = results.train_horizon.unique()
horizons = [60,80]
#print(horizons)
models = results.model_name.unique()
#print(models)
subjects = results.id.unique()  

#model_set = ['2ni','3ni','4ni','2i','3i','4i']
model_set = ["2ni","2ni_lowvar","2ni_lowvar_fit"]
fig1,ax1 = plt.subplots(1,len(horizons),figsize=(15,6))
fig2,ax2 = plt.subplots(1,len(horizons),figsize=(15,6))
for i,horizon in enumerate(horizons):
    #ax1[i].set_title("Training Horizon: {}".format(horizon))
    #ax1[i].set_title("Training Horizon: {}".format(horizon))
    #sample_model = model_set[0]
    #sample_df = results.query("train_horizon==@horizon & model_name == @sample_model")
    #sample_count = sample_df.shape[0]
    #sample_true = sample_df.actual.to_numpy()
    #sample_count_lapse = sum(sample_true)
    for model in model_set:
        filt_df = results.query("train_horizon==@horizon & model_name == @model")
        if filt_df.shape[0]==0:
            continue
        #count = filt_df.shape[0]
        true_val = filt_df.actual.to_numpy()
        #count_lapse = sum(true_val)
        pred_val = filt_df.clip_pred.to_numpy()
        #ax1[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        #ax2[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        metrics.RocCurveDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax1[i],name=model)
        metrics.PrecisionRecallDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax2[i],name=model)

fig1.suptitle('Same-day AUC performance of different models using training horizons of 40, 60, and 80')
fig2.suptitle('Same-day AUCPR performance of different models using training horizons of 40, 60, and 80')
fig1.tight_layout()
fig2.tight_layout()
#fig1.savefig('multi_model_auc_eval.pdf',bbox_inches='tight',facecolor='w')
#fig2.savefig('multi_model_aucpr_eval.pdf',bbox_inches='tight',facecolor='w')
None

In [ ]:
horizons = results.train_horizon.unique()
#print(horizons)
models = results.model_name.unique()
#print(models)
subjects = results.id.unique()  

model_set = ['2ni']
fig1,ax1 = plt.subplots(1,len(horizons),figsize=(15,6))
fig2,ax2 = plt.subplots(1,len(horizons),figsize=(15,6))
for i,horizon in enumerate(horizons):
    ax1[i].set_title("Training Horizon: {}".format(horizon))
    ax1[i].set_title("Training Horizon: {}".format(horizon))
    sample_model = model_set[0]
    sample_df = results.query("train_horizon==@horizon & model_name == @sample_model")
    sample_count = sample_df.shape[0]
    sample_true = sample_df.actual.to_numpy()
    sample_count_lapse = sum(sample_true)
    for model in model_set:
        filt_df = results.query("train_horizon==@horizon & model_name == @model")
        #count = filt_df.shape[0]
        true_val = filt_df.actual.to_numpy()
        #count_lapse = sum(true_val)
        pred_val = filt_df.clip_pred.to_numpy()
        ax1[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        ax2[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        metrics.RocCurveDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax1[i],name=model)
        metrics.PrecisionRecallDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax2[i],name=model)

fig1.suptitle('Same-day AUC performance of model 2ni using training horizons of 40, 60, and 80')
fig2.suptitle('Same-day AUCPR performance of model 2ni using training horizons of 40, 60, and 80')
fig1.tight_layout()
fig2.tight_layout()
fig1.savefig('2ni_auc_eval.pdf',bbox_inches='tight',facecolor='w')
fig2.savefig('2ni_aucpr_eval.pdf',bbox_inches='tight',facecolor='w')
None

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,5))
model_order = ['2ni','3ni','4ni','9ni','2i','3i']
sns.lineplot(ax=ax[0],data=proc_results,x='horizon',y='auc',hue='model',units='model',hue_order=model_order,estimator=None,marker='.',markersize=15)
ax[0].set_title("AUC")
sns.lineplot(ax=ax[1],data=proc_results,x='horizon',y='aucpr',hue='model',units='model',hue_order=model_order,estimator=None,marker='.',markersize=15)
ax[1].set_title("AUCPR")

In [ ]:
coef_results_list=["model_eval/2ni_coef_covar_x.csv","model_eval/2ni_coef_covar_y.csv","model_eval/2ni_coef_covar_no.csv","model_eval/2ni_coef_covar_lapse.csv"]
pred_results_list=["model_eval/2ni_pred_covar_x.csv","model_eval/2ni_pred_covar_y.csv","model_eval/2ni_pred_covar_no.csv","model_eval/2ni_pred_covar_lapse.csv"]
imports=[]
for file in pred_results_list:
    loaded_csv = pd.read_csv(file)
    imports.append(loaded_csv)
window_results = pd.concat(imports,ignore_index=True)
imports=[]
for file in coef_results_list:
    loaded_csv = pd.read_csv(file)
    imports.append(loaded_csv)
coef_results = pd.concat(imports,ignore_index=True)
def clipper(val):
    eps = 10**(-7)
    if val > 1:
        return 1-eps
    elif val < 0:
        return 0+eps
    else:
        return val
    
#pred_results['clip_pred']=pred_results.apply(lambda row: clipper(row.pred),axis=1)
#display(pred_results)

In [ ]:
raw_horizons = pred_results.train_horizon.unique()
horizons= np.delete(raw_horizons,np.where(raw_horizons==90))
print(horizons)
models = pred_results.model_name.unique()
model='2ni'
subjects = pred_results.id.unique()  
covar_list = ['x','y','no','lapse']

model_set = ['2ni']
fig1,ax1 = plt.subplots(1,len(horizons),figsize=(15,6))
fig2,ax2 = plt.subplots(1,len(horizons),figsize=(15,6))
for i,horizon in enumerate(horizons):
    ax1[i].set_title("Training Horizon: {}".format(horizon))
    ax1[i].set_title("Training Horizon: {}".format(horizon))
    sample_model = model_set[0]
    sample_df = pred_results.query("train_horizon==@horizon & model_name == @sample_model & covar=='x'")
    sample_count = sample_df.shape[0]
    sample_true = sample_df.actual.to_numpy()
    sample_count_lapse = sum(sample_true)
    for covar in covar_list:
        filt_df = pred_results.query("train_horizon==@horizon & model_name == @model & covar==@covar")
        #count = filt_df.shape[0]
        true_val = filt_df.actual.to_numpy()
        #count_lapse = sum(true_val)
        pred_val = filt_df.clip_pred.to_numpy()
        ax1[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        ax2[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        metrics.RocCurveDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax1[i],name=covar)
        metrics.PrecisionRecallDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax2[i],name=covar)

fig1.suptitle('Same-day AUC performance of model 2ni using training horizons of 40, 60, and 80')
fig2.suptitle('Same-day AUCPR performance of model 2ni using training horizons of 40, 60, and 80')
fig1.tight_layout()
fig2.tight_layout()
fig1.savefig('2ni_auc_eval.pdf',bbox_inches='tight',facecolor='w')
fig2.savefig('2ni_aucpr_eval.pdf',bbox_inches='tight',facecolor='w')
None

In [ ]:
coef_results.head()
selected_covar = 'no'
coef_selection_df = coef_results.query("train_horizon == 90 & covar == @selected_covar")

In [ ]:
# Hidden State B diagonals
# Convention is that the Z column vector with largest l2 norm corresponds to the x11 hidden state
sns.jointplot(data=coef_selection_df,x='b11',y='b22',xlim=(-0.25,1.25),ylim=(-0.25,1.25))
print('E[b11] =',np.mean(coef_selection_df.b11))
print('E[b22] =',np.mean(coef_selection_df.b22))
None


In [ ]:
# U (or R) offset values
for i in [1,2]:
    str1 = 'u'+str(i)
    fig,ax=plt.subplots(figsize=(6,6))
    sns.histplot(data=coef_selection_df,x=str1,ax=ax,stat='proportion')
    ax.set_title('Distribution of {}'.format(str1))
    print("E[{}] = ".format(str1),np.mean(coef_selection_df[str1]))

In [ ]:
# Elements of Z (EMA and lapse observation matrix)
for i in [1,2,3,4,5,6,7,8,9,10]:
    str1 = 'z'+str(i)+'1'
    str2 = 'z'+str(i)+'2'
    #print(coef_results.query("train_horizon in [90]")[str1].min())
    #print(coef_results.query("train_horizon in [90]")[str2].min())
    minval = min(coef_selection_df[str1].min(),coef_results.query("train_horizon in [90]")[str2].min())
    maxval = max(coef_selection_df[str1].max(),coef_results.query("train_horizon in [90]")[str2].max())
    #lims=(minval-.1,maxval+.1)
    lims=(-1.5,2.5)
    sns.jointplot(data=coef_selection_df,x=str1,y=str2,xlim=lims,ylim=lims)
    print("E[{}] = ".format(str1),np.mean(coef_selection_df[str1]))
    print("E[{}] = ".format(str2),np.mean(coef_selection_df[str2]))
    print("---")
None

In [ ]:
# A offset values
for i in [1,2,3,4,5,6,7,8,9,10]:
    str1 = 'a'+str(i)
    fig,ax=plt.subplots(figsize=(6,6))
    sns.histplot(data=coef_selection_df,x=str1,ax=ax,stat='proportion')
    ax.set_title('Distribution of {}'.format(str1))
    print("E[{}] = ".format(str1),np.mean(coef_selection_df[str1]))
    print("---")

In [ ]:
# Distribution of observation variances
for i in [1,2,3,4,5,6,7,8,9]:
    str1 = 'r'+str(i)
    fig,ax=plt.subplots(figsize=(6,6))
    sns.histplot(data=coef_selection_df,x=str1,ax=ax,stat='proportion')
    ax.set_title('Distribution of {}'.format(str1))
    print("E[{}] = ".format(str1),np.mean(coef_selection_df[str1]))
    print("---")

In [ ]:
#window_results_list = ['model_eval/2ni_pred_covar_no_window_full.csv']
window_results_list = ['model_eval/2ni/pred_nocovar_window.csv']
imports =[]
for file in window_results_list:
    loaded_csv = pd.read_csv(file)
    imports.append(loaded_csv)
window_results = pd.concat(imports,ignore_index=True)

In [ ]:
window_results.head()

In [ ]:
model_set = ['2ni']
horizons = [40,60,80]
window_list = [0,1,2,3,4,5,6,7]

fig1,ax1 = plt.subplots(1,len(horizons),figsize=(15,6))
fig2,ax2 = plt.subplots(1,len(horizons),figsize=(15,6))
fig3,ax3 = plt.subplots(1,len(horizons),figsize=(15,6))
summary_list = []
for i,horizon in enumerate(horizons):
    accuracy_list=[]
    ax1[i].set_title("Training Horizon: {}".format(horizon))
    ax2[i].set_title("Training Horizon: {}".format(horizon))
    #sample_model = model_set[0]
    #sample_df = window_results.query("train_horizon==@horizon")
    #sample_true = window_results['w']
    #sample_count = sample_df.shape[0]
    #sample_true = sample_df.actual.to_numpy()
    #sample_count_lapse = sum(sample_true)
    for window in window_list:
        pred_str = 'w'+str(window)+'_pred'
        act_str = 'w'+str(window)+'_act'
        filt_df = window_results.query("train_horizon==@horizon")
        #count = filt_df.shape[0]
        
        true_val = filt_df[act_str].dropna().to_numpy()
        #count_lapse = sum(true_val)
        pred_val = filt_df[pred_str].dropna().to_numpy()
        fprs,tprs,thresholds = metrics.roc_curve(true_val,pred_val)
        max_acc = -np.inf
        ma_tpr = np.nan
        ma_fpr = np.nan
        ma_thresh = np.nan
        for tpr,fpr,threshold in zip(tprs,fprs,thresholds):
            y_pred = (pred_val>=threshold).astype(int)
            #accuracy = metrics.accuracy_score(true_val,y_pred)
            accuracy = metrics.balanced_accuracy_score(true_val,y_pred)
            iter_df = pd.DataFrame({'horizon':horizon,'threshold':threshold,'acc':accuracy,'window':window,'fpr':fpr,'tpr':tpr},index=[0])
            if threshold!=np.inf:
                accuracy_list.append(iter_df)
                if accuracy>=max_acc:
                    max_acc=accuracy
                    ma_tpr=tpr
                    ma_fpr=fpr
                    ma_thresh=threshold
        horizon_window_df = pd.DataFrame({'horizon':horizon,'window':window,'tpr':ma_tpr,'fpr':ma_fpr,'accuracy':max_acc,'threshold':ma_thresh},index=[0])
        summary_list.append(horizon_window_df)
        #print(window,len(true_val),len(pred_val))
        #ax1[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        #ax2[i].set_title("Horizon: {}, Samples: {}, Lapse fraction: {:.2f}".format(horizon,sample_count,sample_count_lapse/sample_count))
        metrics.RocCurveDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax1[i],name=str(window))
        metrics.PrecisionRecallDisplay.from_predictions(true_val, pred_val,pos_label=1,ax=ax2[i],name=str(window))

    df = pd.concat(accuracy_list,ignore_index=True)
    sns.lineplot(data=df.query("horizon==@horizon"),x='threshold',y='acc',estimator=None,units='window',hue='window',hue_order=window_list,ax=ax3[i],palette='viridis')
fig1.suptitle('n-day Window Prediction AUC performance of model 2ni using training horizons of 40, 60, and 80')
fig2.suptitle('n-day Window Prediction AUCPR performance of model 2ni using training horizons of 40, 60, and 80')
fig1.tight_layout()
fig2.tight_layout()
#fig1.savefig('2ni_auc_window.pdf',bbox_inches='tight',facecolor='w')
#fig2.savefig('2ni_aucpr_window.pdf',bbox_inches='tight',facecolor='w')
summary_df = pd.concat(summary_list,ignore_index=True)
None

In [ ]:
#summary_df.to_csv('max_acc_tpr_fpr.csv')
summary_df.to_csv('balanced_acc_tpr_fpr.csv')

In [ ]:
display(summary_df)

In [ ]:
df.window.unique()

In [ ]:
filt_df = window_results.query("train_horizon==80")
filt_df_sub=filt_df[['pred_t','w0_pred','pred']]
def clipper(val):
    eps = 10**(-12)
    if val > 1-eps:
        return 1-eps
    elif val < eps:
        return 0+eps
    else:
        return val
    
filt_df_sub['clip_pred']=filt_df_sub.apply(lambda row: clipper(row.pred),axis=1)
filt_df_sub['diff']=filt_df_sub.apply(lambda row: row.w0_pred-row.clip_pred,axis=1)
display(filt_df_sub)
idx=filt_df_sub['diff'].idxmax()
filt_df.loc[idx-6:idx+5]

In [ ]:
filt_df_sub.reset_index(inplace=True)
filt_df_sub.head()

In [ ]:
sns.lineplot(x=filt_df_sub.index,y=filt_df_sub['diff'])